# Notebook 3 — Classical baselines

Compute the exact optimum (brute force) and three classical approximations to benchmark against QAOA:

1. **Random cut** — expected ratio 0.5 (trivial baseline)
2. **Greedy heuristic** — fast, often near-optimal on small graphs
3. **Goemans-Williamson SDP relaxation** — best polynomial-time classical approximation, ratio $\ge 0.878$

In [ ]:
import sys; sys.path.insert(0, '../src')
from max_cut import MaxCut
from classical import all_baselines, brute_force_optimal, goemans_williamson
import matplotlib.pyplot as plt

mc = MaxCut.from_edges(5, [(0,1),(1,2),(2,0),(1,3),(3,4),(4,0)], name='5-node test')
print(mc.summary())

## 1. Brute force (exact optimum)

Exhaustive enumeration of all $2^n$ bitstrings. Feasible for $n \le 20$.

In [ ]:
opt_cut, opt_z = brute_force_optimal(mc)
print(f'Optimal cut: {opt_cut} (bitstring: {opt_z})')
mc.draw(assignment=opt_z, title=f'Optimal cut ({opt_cut}/{mc.m} edges)')

## 2. Greedy heuristic

Iterate over nodes in random order; assign each to the side that cuts more already-assigned edges. $O(n+m)$ runtime.

In [ ]:
from classical import greedy_cut
greedy_val, greedy_z = greedy_cut(mc, seed=42)
print(f'Greedy cut: {greedy_val} (bitstring: {greedy_z})')
print(f'Approximation ratio: {greedy_val / opt_cut:.3f}')

## 3. Goemans-Williamson SDP relaxation

Solve the SDP

$$\max_{V} \sum_{(i,j)\in E}\tfrac{1}{2}(1 - \langle v_i, v_j\rangle) \quad \text{s.t.} \quad \|v_i\|^2 = 1 \;\forall i,\; V \succeq 0,$$

then round via a random hyperplane: pick a random $g \in \mathbb{R}^n$, assign node $i$ to side 0 if $\langle g, v_i\rangle \ge 0$ and side 1 otherwise.

**Theorem (Goemans-Williamson 1995)**: $\mathbb{E}[\text{cut}] \ge 0.878 \cdot \text{OPT}$.

In [ ]:
gw_val, gw_z = goemans_williamson(mc, seed=42)
print(f'GW cut: {gw_val} (bitstring: {gw_z})')
print(f'Approximation ratio: {gw_val / opt_cut:.3f}')
print(f'Theoretical lower bound: 0.878')

## 4. All baselines summary

In [ ]:
baselines = all_baselines(mc, seed=42)
for name, res in baselines.items():
    if res['value'] is not None:
        print(f'  {name:25s}: cut = {res["value"]:3d}  (ratio {res["approx_ratio"]:.3f})')
    else:
        print(f'  {name:25s}: skipped ({res.get("note", "")}')